In [1]:
%pip install -r requirements.txt

  Using cached starlette-1.10.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached opentelemetry_api-1.45.2-py3-none-any.whl.metadata (1.4 kB)
  Using cached uvicorn-0.54.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached truststore-0.10.5-py3-none-any.whl.metadata (4.5 kB)
  Using cached python_dotenv-1.2.4-py3-none-any.whl.metadata (29 kB)
Using cached opentelemetry_api-1.45.2-py3-none-any.whl (60 kB)
Using cached python_dotenv-1.2.4-py3-none-any.whl (23 kB)
   ---------------------------------------- 0.0/730.9 kB ? eta -:--:--
   -------------- ------------------------- 262.1/730.9 kB ? eta -:--:--
   ---------------------------------------- 730.9/730.9 kB 2.0 MB/s  0:00:00
   ---------------------------------------- 0.0/555.2 kB ? eta -:--:--
   ------------------ --------------------- 262.1/555.2 kB ? eta -:--:--
   ---------------------------------------- 555.2/555.2 kB 1.9 MB/s  0:00:00
Using cached starlette-1.10.0-py3-none-any.whl (80 kB)
Using cached truststore-0.10.5-py3-non

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

## Step 1 — Packages install karein (sirf ek baar)

In [ ]:
%pip install -r requirements.txt

## Step 2 — Apne computer par chalayein (Supabase ke bina)
Is mode mein data local `data/` folder mein rehta hai. Demo login: `Student@gmail.com / Student@123`, `Admin@gmail.com / Admin@123`.

In [ ]:
import subprocess, sys, os, time
server = None

def start(port=8000):
    global server
    if server and server.poll() is None:
        print("Already running"); return
    server = subprocess.Popen([sys.executable, "-m", "uvicorn", "main:app", "--port", str(port)])
    time.sleep(3)
    print(f"✅ Open this link in your browser:  http://127.0.0.1:{port}")

start()

In [ ]:
# Server band karne ke liye (jab kaam ho jaye)
if server and server.poll() is None:
    server.terminate(); print("Stopped")

## Step 3 — Supabase (free cloud database + file storage) setup
1. **supabase.com** → Sign up (free) → **New project** (naam: `university-club`, database password yaad rakhein).
2. Project ready hone par left menu **SQL Editor → New query** → neeche wale cell ka output copy karke paste karein → **Run**.
3. **Project Settings → API** mein se ye 2 cheezein copy karein:
   - **Project URL** (`https://xxxx.supabase.co`)
   - **service_role** key (⚠️ secret hai, kisi ko mat dikhana)

In [ ]:
print(open("supabase/schema.sql", encoding="utf-8").read())

## Step 4 — `.env` file banayein
Ye cell `.env.example` se `.env` bana dega. Fir VS Code mein **`.env` kholkar** apni values bharein (URL, service key, aapka admin email + password), file save karein.

In [ ]:
import shutil, os
if not os.path.exists(".env"):
    shutil.copy(".env.example", ".env"); print(".env created - ab use kholkar values bharein")
else:
    print(".env already exists")

In [ ]:
# Connection test (Step 4 ke baad). Pehle Step 2 ka stop cell chalayein, phir ye.
import core, store
core.load_env()
s = store.get_store()
print("Mode:", s.kind)
s.init()                       # pehla admin bana dega (ADMIN_IDENTIFIER / ADMIN_PASSWORD se)
print("Users:", s.count_users(), "| Materials:", len(s.list_materials()))

In [ ]:
# Ab Supabase ke saath local run karke check karein
start()

## Step 5 — FastAPI Cloud par deploy
1. **fastapicloud.com** par free account banayein (pricing/plan wahan check karein).
2. Neeche wala cell chalayein — browser khulega, login karein, app ban jayega aur deploy hoga.
   *(Agar cell mein dikkat aaye to VS Code ke Terminal mein `fastapi deploy` chalayein.)*

In [ ]:
!fastapi deploy

In [ ]:
import subprocess
env = {}
for line in open(".env", encoding="utf-8"):
    line = line.strip()
    if line and not line.startswith("#") and "=" in line:
        k, v = line.split("=", 1); env[k.strip()] = v.strip().strip('"').strip("'")

SECRETS = {"SUPABASE_SERVICE_KEY", "ADMIN_PASSWORD", "SECRET_KEY"}
for k, v in env.items():
    if not v: continue
    cmd = ["fastapi", "cloud", "env", "set"] + (["--secret"] if k in SECRETS else []) + [k, v]
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(k, "->", "OK" if r.returncode == 0 else r.stderr.strip()[:200])
subprocess.run(["fastapi", "cloud", "env", "set", "UC_HTTPS_ONLY", "1"])

In [ ]:
!fastapi deploy